# Market Data — Russell 1000 Universe

Fetches daily OHLCV bars from Alpaca for every ticker in `data/ru1000.yml`
and stores them via `MarketDataCache` as:

```
data/historical-prices/<TICKER>/<YEAR>.csv
```

Runs in batches of 50 symbols per API request.  
Already-stored files are skipped so the cell can be re-run safely.

In [1]:
from sentiment.log import setup_logging
setup_logging()

In [2]:
from datetime import datetime
from pathlib import Path

import yaml

from sentiment.sources.alpaca import AlpacaSource
from sentiment.sources.cache import MarketDataCache

In [3]:
with open("../data/ru1000.yml") as f:
    companies = yaml.safe_load(f)["companies"]

tickers = list(companies.keys())

YEARS      = [2018, 2019, 2020]
TIMEFRAME  = "1Day"
BATCH_SIZE = 50   # symbols per Alpaca request

print(f"Tickers   : {len(tickers)}")
print(f"Years     : {YEARS}")
print(f"Batch size: {BATCH_SIZE}")

Tickers   : 992
Years     : [2018, 2019, 2020]
Batch size: 50


In [4]:
alpaca = AlpacaSource()
cache  = MarketDataCache()

for year in YEARS:
    start = datetime(year, 1, 1)
    end   = datetime(year, 12, 31)
    n_batches = (len(tickers) + BATCH_SIZE - 1) // BATCH_SIZE
    print(f"\n=== {year} ({start.date()} \u2192 {end.date()}) — {n_batches} batches ===")

    stored  = 0
    skipped = 0
    errors  = 0

    for i in range(0, len(tickers), BATCH_SIZE):
        batch = tickers[i : i + BATCH_SIZE]

        # Skip tickers whose <TICKER>/<YEAR>.csv already exists
        to_fetch = [t for t in batch if not cache.is_cached(t, year)]
        skipped += len(batch) - len(to_fetch)

        if not to_fetch:
            continue

        try:
            df = alpaca.fetch_bars(to_fetch, TIMEFRAME, start, end)
        except Exception as exc:
            print(f"  [ERROR] batch {i // BATCH_SIZE + 1}: {exc}")
            errors += len(to_fetch)
            continue

        if df.empty:
            continue

        for ticker, group in df.groupby("symbol"):
            cache.store(ticker, year, group)
            stored += 1

        batch_num = i // BATCH_SIZE + 1
        print(f"  batch {batch_num:3d}/{n_batches}  stored={stored}  skipped={skipped}  errors={errors}")

    print(f"Year {year} complete — stored: {stored}, skipped: {skipped}, errors: {errors}")

print("\nAll years complete.")


=== 2018 (2018-01-01 → 2018-12-31) — 20 batches ===
Year 2018 complete — stored: 0, skipped: 873, errors: 0

=== 2019 (2019-01-01 → 2019-12-31) — 20 batches ===
Year 2019 complete — stored: 0, skipped: 895, errors: 0

=== 2020 (2020-01-01 → 2020-12-31) — 20 batches ===
Year 2020 complete — stored: 0, skipped: 926, errors: 0

All years complete.
